<a href="https://colab.research.google.com/github/YoussefMedhat2200626/Distributed-Online-Marketplace/blob/main/youtuberplaylistcleaner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 🔄 Channel-to-Channel Transfer (Warning Channel safety in on your own Risk dont transfer copyrighted videos)
!pip install google-api-python-client google-auth-oauthlib google-auth-httplib2 yt-dlp -q
!pip install --upgrade yt-dlp

import os
import glob
import subprocess
import yt_dlp
import googleapiclient.discovery
import googleapiclient.errors
from googleapiclient.http import MediaFileUpload
from google_auth_oauthlib.flow import InstalledAppFlow
from google.colab import userdata
import json # Import json module


os.environ['OAUTHLIB_INSECURE_TRANSPORT'] = '1'

# Retrieve the client secrets JSON string from Colab userdata
client_secrets_json_str = userdata.get('client_secrets')

# Define the name for the temporary client secrets file
CLIENT_SECRETS_FILE_NAME = 'temp_client_secrets.json'

# Write the client secrets JSON string to a temporary file
try:
    with open(CLIENT_SECRETS_FILE_NAME, 'w') as f:
        f.write(client_secrets_json_str)
except Exception as e:
    print(f"Error writing client secrets to file: {e}")
    # Handle error or exit if file creation is critical

CLIENT_SECRETS_FILE = CLIENT_SECRETS_FILE_NAME
SCOPES = ["https://www.googleapis.com/auth/youtube"]
API_SERVICE_NAME = "youtube"
API_VERSION = "v3"

# ==========================================
# 🖇️ COLAB UI SETTINGS
# ==========================================
# @markdown ### 🔗 Playlist & Upload Settings
source_playlist_url = "https://youtube.com/playlist?list=PLdUZbyW_EOzq282is5uGheG9nDxmwBsTB&si=py1jmjlEWIWB0j-1" # @param {type:"string"}
privacy_status = "unlisted" # @param ["public", "unlisted", "private"]

# @markdown ---
# @markdown ### 📜 Visual Edit Settings (FFmpeg)
apply_blackout = False # @param {type:"boolean"}
# @markdown *(Enable 12% Blackout to Right Side - Hides Teams Bar)*

enable_crop = True # @param {type:"boolean"}
cut_left_percent = 0 # @param {type:"slider", min:0, max:50, step:1}
cut_right_percent = 12 # @param {type:"slider", min:0, max:50, step:1}
cut_top_percent = 6 # @param {type:"slider", min:0, max:50, step:1}
cut_bottom_percent = 6 # @param {type:"slider", min:0, max:50, step:1}
# ==========================================

def get_authenticated_service():
    flow = InstalledAppFlow.from_client_secrets_file(CLIENT_SECRETS_FILE, SCOPES)
    flow.redirect_uri = 'http://localhost'
    auth_url, _ = flow.authorization_url(prompt='consent')
    print("⚠️ Google Auth Setup (SELECT YOUR DESTINATION CHANNEL):\n")
    print(f"1️⃣ Click this link and authorize the app:\n\n{auth_url}\n")
    print("2️⃣ After allowing access, your browser will say 'This site can't be reached'.")
    print("3️⃣ Copy the ENTIRE URL from your browser's address bar.")
    auth_response = input("\n👇 Paste that full localhost URL right here and hit Enter:\n")
    flow.fetch_token(authorization_response=auth_response.strip())
    return googleapiclient.discovery.build(API_SERVICE_NAME, API_VERSION, credentials=flow.credentials)

def upload_video(youtube, file_path, title, description, privacy):
    print(f"   🔝 Uploading to Destination: {title}")

    file_size = os.path.getsize(file_path)
    print(f"   📁 File: {file_path} | Size: {file_size / (1024*1024):.2f} MB")
    if file_size == 0:
        raise Exception("File is 0 bytes.")

    body = {
        "snippet": {
            "title": (title or 'Untitled')[:100],
            "description": (description or '')[:5000],
            "categoryId": "27"
        },
        "status": {"privacyStatus": privacy}
    }

    media = MediaFileUpload(file_path, chunksize=5 * 1024 * 1024, resumable=True, mimetype="video/mp4")

    request = youtube.videos().insert(
        part="snippet,status",
        body=body,
        media_body=media
    )

    response = None
    while response is None:
        status, response = request.next_chunk()
        if status:
            print(f"   🔝️ Uploading {int(status.progress() * 100)}%...", end="\r")

    print(f"\n   ✅ Video Uploaded! ID: {response['id']}")
    return response['id']

# --- Main ---
try:
    youtube_dest = get_authenticated_service()

    ydl_opts_extract = {
        'extract_flat': True,
        'quiet': True,
        'extractor_args': {'youtubetab': ['skip=authcheck']} # Bypass YouTube bot walls
    }

    print("\n🔍 Fetching videos from Source Playlist...")
    with yt_dlp.YoutubeDL(ydl_opts_extract) as ydl:
        playlist_info = ydl.extract_info(source_playlist_url, download=False)

    if 'entries' not in playlist_info:
        print("❌ Could not find videos in the provided URL.")
    else:
        videos = playlist_info['entries']
        print(f"Found {len(videos)} videos to transfer.")

        for index, video in enumerate(videos):
            video_url = video['url']
            original_title = video.get('title', f'Transferred_Video_{index}')
            original_desc = video.get('description', '')

            print(f"\n[{index+1}/{len(videos)}] 🔻 Downloading: {original_title}")

            # Cleanup old temp files
            for f in glob.glob("temp_*"):
                os.remove(f)

            ydl_opts_download = {
                'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
                'outtmpl': 'temp_raw.%(ext)s',
                'quiet': True,
                'no_warnings': True,
                'merge_output_format': 'mp4',
            }

            try:
                with yt_dlp.YoutubeDL(ydl_opts_download) as ydl:
                    ydl.download([video_url])

                matches = glob.glob("temp_raw.*")
                actual_file = matches[0] if matches else None
                target_file = actual_file

                if actual_file and os.path.exists(actual_file):

                    # --- 📜 APPLY VISUAL EDITS (HUGGING FACE LOGIC) ---
                    filters = []
                    if apply_blackout:
                        filters.append("drawbox=x=iw-iw*(12/100):y=0:w=iw*(12/100):h=ih:color=black:t=fill")
                    if enable_crop and any(v > 0 for v in [cut_left_percent, cut_right_percent, cut_top_percent, cut_bottom_percent]):
                        cl, cr, ct, cb = cut_left_percent, cut_right_percent, cut_top_percent, cut_bottom_percent
                        filters.append(f"crop='iw*(100-{cl}-{cr})/100:ih*(100-{ct}-{cb})/100:iw*{cl}/100:ih*{ct}/100'")

                    if filters:
                        print(f"   ✂️ Applying Visual Edits (Encoding... this will take a few minutes)")
                        processed_file = "temp_processed.mp4"
                        ff_cmd = [
                            "ffmpeg", "-y", "-i", actual_file,
                            "-vf", ",".join(filters),
                            "-c:v", "libx264", "-preset", "ultrafast", "-crf", "22",
                            "-c:a", "copy", "-nostats", "-loglevel", "warning", processed_file
                        ]

                        try:
                            subprocess.run(ff_cmd, check=True)
                            target_file = processed_file
                            print("   ✅ Filtering complete.")
                        except Exception as e:
                            print(f"   ❌ Filtering failed. Falling back to original video. Error: {str(e)}")
                            target_file = actual_file
                    # --------------------------------------------------

                    # Upload whatever the final target_file is
                    upload_video(youtube_dest, target_file, original_title, original_desc, privacy_status)

                    # Cleanup
                    for f in glob.glob("temp_*"):
                        os.remove(f)
                    print("   🧹 Cleaned up temporary files.")
                else:
                    print("   ❌ No file found after download — skipping.")

            except googleapiclient.errors.HttpError as e:
                print(f"   ❌ YouTube API Error (quota limit?): {e}")
                break
            except Exception as e:
                print(f"   ❌ Error processing '{original_title}': {e}")
                for f in glob.glob("temp_*"):
                    os.remove(f)

        print("\n🎉 Transfer session completed!")

except Exception as e:
    print(f"\n❌ Fatal error: {e}")
finally:
    # Clean up the temporary client secrets file
    if os.path.exists(CLIENT_SECRETS_FILE_NAME):
        os.remove(CLIENT_SECRETS_FILE_NAME)
        print(f"Cleaned up {CLIENT_SECRETS_FILE_NAME}.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 39.6 MB/s eta 0:00:00
⚠️ Google Auth Setup (SELECT YOUR DESTINATION CHANNEL):

1️⃣ Click this link and authorize the app:

https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=1048052408535-75cd7s67445t0s8a5pbatr00bjksl44c.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fyoutube&state=YZVYNg2YWtbgXzn1eyeeSvoJXIOFu3&code_challenge=yOiUZZM8rVz0hFvdReGyhqfDGQafug2LkT3ilbluMRs&code_challenge_method=S256&prompt=consent&access_type=offline

2️⃣ After allowing access, your browser will say 'This site can't be reached'.
3️⃣ Copy the ENTIRE URL from your browser's address bar.

👇 Paste that full localhost URL right here and hit Enter:
http://localhost/?state=YZVYNg2YWtbgXzn1eyeeSvoJXIOFu3&iss=https://accounts.google.com&code=4/0AdkVLPwo4iXDmmcnHhcrjALpNa4BLBQ_bu0r1QeqS7Uqq3mlbPhAHGS